## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 07: Walk-Forward Predictability (Secondary, Null Result)**

---

**Scope:** Tests whether the settlement-weekday reforms created exploitable next-day return predictability, using a leakage-free five-fold expanding-window protocol. Reported honestly as a null result.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, expiry calendar, holidays, F&O Bhavcopy archive
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical/volatility features; lot-size-invariant turnover and open interest
3. **03_weekly_options_discontinuation_identification.ipynb** -- dating the Nov 2024 weekly-contract removal directly from exchange records
4. **04_primary_diff_in_diff_analysis.ipynb** -- the headline difference-in-differences result (Bank Nifty vs. Nifty 50)
5. **05_robustness_checks.ipynb** -- placebo cutover dates, leave-one-out, event-study leads/lags
6. **06_expiry_volume_event_study.ipynb** -- descriptive expiry-day volume/OI patterns by regime
7. **07_walk_forward_predictability.ipynb** -- secondary analysis: leakage-free return-predictability test (null result)
8. **08_interrupted_time_series.ipynb** -- secondary analysis: settlement-weekday effect on returns/volatility (null result)
9. **09_figures_for_paper.ipynb** -- regenerates all manuscript figures

**Current location:** **07_walk_forward_predictability.ipynb** <- *you are here*


### Why this is in the paper despite being null

This is the question an expiration-day calendar-effects study would typically ask first. The full five-fold refit (`scripts.walk_forward_backtest`) refits both the regime-detection model and the forecasting model within each fold, so it takes a few minutes to run end-to-end; rebuild with:

```bash
python -m scripts.walk_forward_backtest
```

This notebook loads the saved results rather than re-running the fit.

In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\BN_Structure_Reforms\banknifty-expiry-volatility-ml


In [2]:
with open(config.RESULTS_DIR / 'walk_forward_results.json') as f:
    wf = json.load(f)

conditions = ['Baseline (Technical Only)', 'Baseline + Expiry', 'Baseline + Regime', 'Proposed (Combined)']
rows = []
for cnd in conditions:
    ci = wf['pooled_significance'][f'DirAcc_CI_{cnd}']
    rows.append({'Condition': cnd, 'Accuracy': ci['accuracy_mean'], 'CI_low': ci['ci_2.5'], 'CI_high': ci['ci_97.5']})
pd.DataFrame(rows)

,Condition,Accuracy,CI_low,CI_high
0,Baseline (Technical Only),0.493827,0.444444,0.543210
1,Baseline + Expiry,0.491358,0.444444,0.538333
2,Baseline + Regime,0.518519,0.469136,0.565432
3,Proposed (Combined),0.483951,0.437037,0.533333


In [3]:
dm = wf['pooled_significance']['DM_test_pooled']
sharpe = wf['pooled_significance']['Sharpe_diff_bootstrap']
print(f"Pooled Diebold-Mariano p-value (baseline vs. combined): {dm['P_Value_Two_Sided']:.4f}")
print(f"Pooled Sharpe-ratio difference (combined minus baseline): {sharpe['diff_mean']:.4f} "
      f"[{sharpe['ci_2.5']:.4f}, {sharpe['ci_97.5']:.4f}]")

Pooled Diebold-Mariano p-value (baseline vs. combined): 0.7484
Pooled Sharpe-ratio difference (combined minus baseline): -0.8256 [-2.0126, 0.3119]


Every condition's confidence interval straddles 50% chance accuracy, and the pooled Diebold-Mariano test does not reject equal predictive accuracy. This null result is reported alongside the primary liquidity-concentration finding rather than omitted.